In [1]:
!pip install -q torchmetrics[audio]
!pip install -q librosa requests soundfile pandas tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.6 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/gabrielmittag/NISQA.git
%cd NISQA

!pip install -q torch torchaudio

Cloning into 'NISQA'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 258 (delta 55), reused 45 (delta 45), pack-reused 188 (from 1)
Receiving objects: 100% (258/258), 2.27 MiB | 17.77 MiB/s, done.
Resolving deltas: 100% (130/130), done.
/content/NISQA


In [3]:
import torch
import librosa
import pandas as pd
import numpy as np

from tqdm import tqdm

from torchmetrics.audio import NonIntrusiveSpeechQualityAssessment

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

nisqa = NonIntrusiveSpeechQualityAssessment(
    fs=16000
).to(device)

In [6]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/tts_benchmark/PIPER_FAS/tts_dataset.csv"
)

df.head()

,audio_path,text,tts_runtime,audio_duration,tts_rtf
0,/content/drive/MyDrive/tts_benchmark/PIPER_FAS...,جسد مزبور را در بیشهای که در گودی کوهستان قرار...,2.393545,4.017052,0.595846
1,/content/drive/MyDrive/tts_benchmark/PIPER_FAS...,لیدی گاگای تازه وارد به یکباره,1.498642,2.159456,0.693990
2,/content/drive/MyDrive/tts_benchmark/PIPER_FAS...,رازی را آشکار کردن,0.467736,1.335147,0.350325
3,/content/drive/MyDrive/tts_benchmark/PIPER_FAS...,ماه هاست سکوت کردم,0.398849,1.288707,0.309496
4,/content/drive/MyDrive/tts_benchmark/PIPER_FAS...,چه کسی این نامه را فرستاده است,0.442227,1.637007,0.270144


In [7]:
import pandas as pd

df[["audio_path"]].rename(
    columns={"audio_path": "deg"}
).to_csv(
    "nisqa_input.csv",
    index=False
)

In [8]:
audio_files = df["audio_path"].tolist()

In [10]:
import os

rows = []

for file_path in tqdm(audio_files):

    waveform, sr = librosa.load(file_path, sr=16000)

    waveform = torch.tensor(waveform).unsqueeze(0).to(device)

    with torch.no_grad():
        mos_pred = nisqa(waveform)

    # FIX: handle multi-output tensor
    mos_pred = mos_pred.squeeze().detach().cpu().numpy()

    rows.append({
        "file": os.path.basename(file_path),

        "mos_pred": float(mos_pred[0]),
        "noi_pred": float(mos_pred[1]),
        "dis_pred": float(mos_pred[2]),
        "col_pred": float(mos_pred[3]),
        "loud_pred": float(mos_pred[4]),
    })

100%|██████████| 1052/1052 [14:10<00:00,  1.24it/s]


In [11]:
nisqa_df = pd.DataFrame(rows)

nisqa_df.head()

,file,mos_pred,noi_pred,dis_pred,col_pred,loud_pred
0,00000.wav,4.814481,4.295552,4.883589,4.430002,4.515143
1,00001.wav,3.760830,4.200513,4.267927,3.601216,3.522478
2,00002.wav,4.723557,3.823570,4.799886,4.404103,4.473119
3,00003.wav,4.856455,4.076084,4.850337,4.368119,4.734487
4,00004.wav,4.616113,3.985591,4.700132,4.174289,4.410913


In [12]:
summary = pd.DataFrame([{
    "model": "PIPER_TTS_FAS",
    "samples": len(nisqa_df),

    "nisqa_mos_mean": nisqa_df["mos_pred"].mean(),
    "nisqa_mos_std": nisqa_df["mos_pred"].std(),

    "noise_mean": nisqa_df["noi_pred"].mean(),
    "discontinuity_mean": nisqa_df["dis_pred"].mean(),
    "coloration_mean": nisqa_df["col_pred"].mean(),
    "loudness_mean": nisqa_df["loud_pred"].mean(),
}])

summary

,model,samples,nisqa_mos_mean,nisqa_mos_std,noise_mean,discontinuity_mean,coloration_mean,loudness_mean
0,PIPER_TTS_FAS,1052,4.533503,0.361612,4.070209,4.656351,4.168406,4.28207


In [13]:
import os

SAVE_DIR = "/content/drive/MyDrive/tts_benchmark/PIPER_TTS_FAS"
os.makedirs(SAVE_DIR, exist_ok=True)

In [14]:
SUMMARY_PATH = os.path.join(
    SAVE_DIR,
    "PIPER_TTS_nisqa_summary.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Saved summary:")
print(SUMMARY_PATH)

Saved summary:
/content/drive/MyDrive/tts_benchmark/PIPER_TTS_FAS/PIPER_TTS_nisqa_summary.csv
